### Phase 1: Environment Setup & Dual-Model Loading
In this step, we load TWO distinct models:
1. **Base NLP Model:** Used for basic linguistic tasks (sentence splitting, tokenization).
2. **Custom NER Model:** The machine learning model you trained in `ner_train.ipynb`. This model has learned the *semantic patterns* of hard skills and can extract out-of-vocabulary (OOV) technologies.

In [2]:
import spacy
import time
from dataclasses import dataclass
from typing import List, Set, Dict
import re

# 1. Load Base Model
try:
    nlp_base = spacy.load("en_core_web_sm")
except OSError:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp_base = spacy.load("en_core_web_sm")

# 2. Load Custom Trained NER Model
try:
    # If saved as a spaCy model directory:
    nlp_custom_ner = spacy.load("hard_skill_model.pkl")
    print("✅ Custom ML-NER Model loaded successfully!")
except OSError:
    print("⚠️ Custom model not found at specified path. Using base model as a placeholder for demonstration.")
    nlp_custom_ner = nlp_base # Fallback strictly for code execution

print("✅ Phase 1 Complete. Models are ready.")

⚠️ Custom model not found at specified path. Using base model as a placeholder for demonstration.
✅ Phase 1 Complete. Models are ready.


### Phase 2: True Hybrid Skill Matcher Engine
This class implements the Dual-Engine logic:
* **`_extract_via_era()`**: Uses the predefined dictionary (high precision, low recall).
* **`_extract_via_ner()`**: Uses the custom Machine Learning model to discover new skills (high recall, handles unseen data).
* **`merge_and_score()`**: Fuses the results, removes duplicates, and applies the ERA Priority Score formula ($Relevance \times Difficulty$).

In [3]:
import math
import pandas as pd

# ── Taxonomy: canonical name → { aliases, category } ─────────────────────────
# Difficulty and base_relevance are intentionally removed.
# Difficulty is computed dynamically from JD sentence context (see DIFFICULTY_WEIGHTS).
# Relevance is computed entirely from JD content (tf, position, signal).

TAXONOMY: Dict[str, dict] = {

    # ── Programming ───────────────────────────────────────────────────────────
    "Python": {
        "aliases": ["python3", "python 3", "py", "python programming", "cpython",
                    "python scripting", "python development"],
        "category": "Programming",
    },
    "SQL": {
        "aliases": ["mysql", "postgresql", "postgres", "sqlite", "t-sql", "pl/sql",
                    "structured query language", "relational database", "oracle sql",
                    "ms sql", "sql server", "transact-sql", "query language"],
        "category": "Database",
    },
    "R": {
        "aliases": ["r programming", "r language", "rstudio", "r studio",
                    "tidyverse", "ggplot2", "dplyr", "caret"],
        "category": "Programming",
    },
    "Java": {
        "aliases": ["java se", "java ee", "jvm", "spring", "spring boot",
                    "spring framework", "maven", "gradle", "j2ee"],
        "category": "Programming",
    },
    "Scala": {
        "aliases": ["scala programming", "akka", "play framework"],
        "category": "Programming",
    },
    "C++": {
        "aliases": ["cpp", "c plus plus", "stl", "cmake"],
        "category": "Programming",
    },
    "Go": {
        "aliases": ["golang", "go programming", "go language"],
        "category": "Programming",
    },
    "JavaScript": {
        "aliases": ["js", "node.js", "nodejs", "node js", "es6", "typescript",
                    "react", "vue", "angular"],
        "category": "Programming",
    },

    # ── Machine Learning / AI ─────────────────────────────────────────────────
    "Machine Learning": {
        "aliases": ["ml", "statistical modeling", "statistical modelling",
                    "predictive modeling", "predictive modelling",
                    "supervised learning", "unsupervised learning",
                    "semi-supervised", "regression", "classification",
                    "clustering", "ml algorithms", "machine learning algorithms",
                    "ml models", "ml pipeline", "ml workflow"],
        "category": "ML/AI",
    },
    "Deep Learning": {
        "aliases": ["dl", "neural network", "neural networks", "ann",
                    "cnn", "convolutional neural network", "rnn",
                    "recurrent neural network", "lstm", "gru",
                    "transformer", "attention mechanism", "attention model",
                    "self-attention", "encoder decoder", "autoencoder",
                    "generative model", "diffusion model"],
        "category": "ML/AI",
    },
    "NLP": {
        "aliases": ["natural language processing", "text mining", "text analytics",
                    "sentiment analysis", "named entity recognition", "ner",
                    "language model", "large language model", "llm", "bert",
                    "gpt", "text classification", "information extraction",
                    "question answering", "text generation", "summarization",
                    "machine translation", "word embedding", "word2vec", "glove",
                    "prompt engineering", "fine-tuning llm", "rag",
                    "retrieval augmented generation"],
        "category": "ML/AI",
    },
    "Computer Vision": {
        "aliases": ["cv", "image recognition", "image classification",
                    "object detection", "yolo", "opencv", "image segmentation",
                    "semantic segmentation", "image processing", "video analysis",
                    "ocr", "optical character recognition"],
        "category": "ML/AI",
    },
    "Reinforcement Learning": {
        "aliases": ["rl", "reward learning", "q-learning", "policy gradient",
                    "actor critic", "dqn", "ppo", "multi-agent"],
        "category": "ML/AI",
    },
    "Generative AI": {
        "aliases": ["genai", "gen ai", "stable diffusion", "dall-e",
                    "gpt-4", "gpt4", "chatgpt", "foundation model",
                    "multimodal", "text to image"],
        "category": "ML/AI",
    },

    # ── ML Frameworks ─────────────────────────────────────────────────────────
    "TensorFlow": {
        "aliases": ["tf", "tensorflow 2", "tensorflow2", "keras", "tf2",
                    "tensorflow serving", "tensorflow lite", "tflite"],
        "category": "ML Framework",
    },
    "PyTorch": {
        "aliases": ["pytorch", "torch", "libtorch", "pytorch lightning",
                    "torchvision", "torchaudio", "torchserve"],
        "category": "ML Framework",
    },
    "Scikit-learn": {
        "aliases": ["sklearn", "scikit learn", "sci-kit learn", "scikit-learn"],
        "category": "ML Framework",
    },
    "XGBoost": {
        "aliases": ["xgb", "gradient boosting", "lgbm", "lightgbm",
                    "catboost", "gbm", "boosting", "gbdt",
                    "gradient boosted trees"],
        "category": "ML Framework",
    },
    "Hugging Face": {
        "aliases": ["huggingface", "hf", "transformers library",
                    "hugging face transformers", "datasets library",
                    "peft", "lora", "qlora", "accelerate"],
        "category": "ML Framework",
    },

    # ── Data Engineering ──────────────────────────────────────────────────────
    "Spark": {
        "aliases": ["apache spark", "pyspark", "spark sql", "spark streaming",
                    "spark ml", "databricks spark", "rdd"],
        "category": "Data Engineering",
    },
    "Kafka": {
        "aliases": ["apache kafka", "kafka streaming", "event streaming",
                    "message queue", "rabbitmq", "confluent", "kafka connect",
                    "kafka streams", "event-driven", "pub sub"],
        "category": "Data Engineering",
    },
    "Airflow": {
        "aliases": ["apache airflow", "workflow orchestration", "dag",
                    "luigi", "prefect", "dagster", "workflow automation"],
        "category": "Data Engineering",
    },
    "dbt": {
        "aliases": ["data build tool", "dbt core", "dbt cloud"],
        "category": "Data Engineering",
    },
    "Snowflake": {
        "aliases": ["snowflake db", "snowflake data warehouse", "snowpark"],
        "category": "Data Engineering",
    },
    "Databricks": {
        "aliases": ["databricks platform", "delta lake", "delta table",
                    "unity catalog", "mlflow databricks"],
        "category": "Data Engineering",
    },
    "Hadoop": {
        "aliases": ["hdfs", "mapreduce", "hive", "hbase", "pig", "yarn",
                    "hadoop ecosystem"],
        "category": "Data Engineering",
    },
    "ETL": {
        "aliases": ["extract transform load", "etl pipeline", "data pipeline",
                    "data ingestion", "data integration", "elt",
                    "data warehouse pipeline", "fivetran", "stitch"],
        "category": "Data Engineering",
    },

    # ── Cloud ─────────────────────────────────────────────────────────────────
    "AWS": {
        "aliases": ["amazon web services", "ec2", "s3", "lambda", "sagemaker",
                    "emr", "redshift", "aws cloud", "cloudformation",
                    "aws glue", "athena", "kinesis", "sns", "sqs",
                    "elastic beanstalk", "ecs", "eks", "aws rds", "dynamodb"],
        "category": "Cloud",
    },
    "GCP": {
        "aliases": ["google cloud", "google cloud platform", "bigquery",
                    "vertex ai", "dataflow", "cloud run", "gke",
                    "cloud storage", "cloud functions", "cloud spanner",
                    "looker", "google analytics"],
        "category": "Cloud",
    },
    "Azure": {
        "aliases": ["microsoft azure", "azure ml", "azure devops",
                    "azure functions", "cosmos db", "azure blob",
                    "azure data factory", "azure synapse", "azure databricks",
                    "aks", "power platform"],
        "category": "Cloud",
    },

    # ── MLOps / DevOps ────────────────────────────────────────────────────────
    "Docker": {
        "aliases": ["containerization", "container", "dockerfile",
                    "docker compose", "docker swarm", "container image",
                    "docker hub"],
        "category": "MLOps/DevOps",
    },
    "Kubernetes": {
        "aliases": ["k8s", "container orchestration", "helm", "kubectl",
                    "kustomize", "kubernetes cluster", "service mesh", "istio"],
        "category": "MLOps/DevOps",
    },
    "MLflow": {
        "aliases": ["ml flow", "experiment tracking", "model registry",
                    "kubeflow", "wandb", "weights and biases",
                    "neptune", "comet", "model versioning"],
        "category": "MLOps",
    },
    "CI/CD": {
        "aliases": ["continuous integration", "continuous deployment",
                    "github actions", "jenkins", "gitlab ci", "circleci",
                    "travis ci", "continuous delivery", "devops pipeline",
                    "automated testing", "deployment pipeline"],
        "category": "DevOps",
    },
    "Terraform": {
        "aliases": ["infrastructure as code", "iac", "pulumi",
                    "terraform cloud", "ansible", "cloudformation"],
        "category": "DevOps",
    },

    # ── Databases ─────────────────────────────────────────────────────────────
    "MongoDB": {
        "aliases": ["mongo", "mongodb atlas", "nosql database", "document database"],
        "category": "Database",
    },
    "Redis": {
        "aliases": ["redis cache", "in-memory database", "caching", "memcached"],
        "category": "Database",
    },
    "Elasticsearch": {
        "aliases": ["elastic search", "elk stack", "kibana", "logstash",
                    "opensearch", "full-text search"],
        "category": "Database",
    },

    # ── Data Analysis / Visualization ─────────────────────────────────────────
    "Data Analysis": {
        "aliases": ["data analytics", "exploratory data analysis", "eda",
                    "statistical analysis", "quantitative analysis",
                    "data exploration", "business analytics",
                    "descriptive analytics"],
        "category": "Analytics",
    },
    "Tableau": {
        "aliases": ["tableau desktop", "tableau server", "tableau public",
                    "tableau prep", "tableau online"],
        "category": "Visualization",
    },
    "Power BI": {
        "aliases": ["powerbi", "power bi desktop", "microsoft power bi",
                    "dax", "power query", "power bi service"],
        "category": "Visualization",
    },
    "Matplotlib": {
        "aliases": ["seaborn", "plotly", "ggplot", "ggplot2",
                    "data visualization", "visualisation", "bokeh",
                    "altair", "dash", "streamlit"],
        "category": "Visualization",
    },
    "Excel": {
        "aliases": ["microsoft excel", "ms excel", "spreadsheet",
                    "pivot table", "vba", "vlookup", "google sheets"],
        "category": "Tools",
    },

    # ── Statistics ────────────────────────────────────────────────────────────
    "Statistics": {
        "aliases": ["statistical methods", "probability", "hypothesis testing",
                    "a/b testing", "ab testing", "bayesian", "frequentist",
                    "experimental design", "causal inference",
                    "multivariate analysis", "time series", "forecasting",
                    "survival analysis", "econometrics"],
        "category": "Statistics",
    },

    # ── Tools ─────────────────────────────────────────────────────────────────
    "Git": {
        "aliases": ["github", "gitlab", "version control", "bitbucket",
                    "git workflow", "pull request", "code review"],
        "category": "Tools",
    },
    "Linux": {
        "aliases": ["unix", "bash", "shell scripting", "command line",
                    "ubuntu", "centos", "debian", "shell script",
                    "bash scripting", "zsh"],
        "category": "Tools",
    },

    # ── Soft Skills ───────────────────────────────────────────────────────────
    "Communication": {
        "aliases": ["presentation skills", "stakeholder communication",
                    "written communication", "verbal communication",
                    "storytelling with data", "data storytelling",
                    "executive communication"],
        "category": "Soft Skills",
    },
}


def build_alias_lookup(taxonomy: dict) -> dict:
    """
    Flatten taxonomy into a single lookup dict.
    Every alias AND the canonical name itself → canonical name.
    e.g.  "sklearn" → "Scikit-learn",  "k8s" → "Kubernetes"
    """
    lookup = {}
    for canonical, info in taxonomy.items():
        lookup[canonical.lower()] = canonical
        for alias in info.get("aliases", []):
            lookup[alias.lower()] = canonical
    return lookup


ALIAS_LOOKUP = build_alias_lookup(TAXONOMY)
print(f"✅ Taxonomy loaded: {len(TAXONOMY)} canonical skills")
print(f"   Alias lookup entries: {len(ALIAS_LOOKUP)}")


✅ Taxonomy loaded: 47 canonical skills
   Alias lookup entries: 446


In [4]:
# ── Importance signal weights (continuous 0–1) ────────────────────────────────
# Used in relevance scoring: how strongly does this sentence signal
# that the skill is genuinely required?
IMPORTANCE_WEIGHTS: Dict[str, float] = {
    "required":      1.00,
    "must":          1.00,
    "mandatory":     1.00,
    "essential":     0.95,
    "critical":      0.95,
    "necessary":     0.90,
    "expert":        0.88,
    "expertise":     0.85,
    "hands-on":      0.82,
    "extensive":     0.80,
    "deep":          0.78,
    "strong":        0.75,
    "solid":         0.72,
    "proficient":    0.70,
    "proficiency":   0.70,
    "experience":    0.55,
    "knowledge":     0.50,
    "background":    0.45,
    "understanding": 0.40,
    "familiarity":   0.30,
    "preferred":     0.20,
    "familiar":      0.18,
    "a plus":        0.12,
    "exposure":      0.12,
    "basic":         0.08,
}

# ── Difficulty modifier weights (continuous 0–1) ──────────────────────────────
# Used in difficulty scoring: how hard does this JD expect you to be at
# the skill? Longer phrases are checked first to avoid partial matches.
DIFFICULTY_WEIGHTS: Dict[str, float] = {
    "mastery":              1.00,
    "expert":               1.00,
    "in-depth":             0.90,
    "deep knowledge":       0.88,
    "deep understanding":   0.85,
    "advanced":             0.85,
    "senior":               0.82,
    "extensive experience": 0.80,
    "years of experience":  0.78,
    "5+ years":             0.82,
    "4+ years":             0.75,
    "3+ years":             0.67,
    "proficient":           0.65,
    "proficiency":          0.65,
    "strong":               0.60,
    "solid":                0.58,
    "good understanding":   0.52,
    "working knowledge":    0.50,
    "2+ years":             0.48,
    "1+ year":              0.36,
    "familiar":             0.32,
    "familiarity":          0.30,
    "some experience":      0.28,
    "exposure":             0.25,
    "a plus":               0.18,
    "preferred":            0.20,
    "nice to have":         0.15,
    "basic":                0.12,
}

# Sentence-level phrases that mark a Requirements / Qualifications block
REQUIREMENTS_PHRASES = [
    "requirement", "requirements", "qualification", "qualifications",
    "must have", "must-have", "you will need", "minimum", "mandatory",
    "what we need", "what you need", "we require", "we are looking for",
]


# ── Helper functions ──────────────────────────────────────────────────────────

def _sentence_signal_score(sentence: str) -> float:
    """
    Returns the highest importance weight found in a sentence.
    0.0 if no importance signal is present.
    """
    s = sentence.lower()
    weights = [w for sig, w in IMPORTANCE_WEIGHTS.items() if sig in s]
    return max(weights) if weights else 0.0


def _is_requirements_sentence(sentence: str) -> bool:
    """True if the sentence appears to be in a Requirements / Qualifications block."""
    s = sentence.lower()
    return any(phrase in s for phrase in REQUIREMENTS_PHRASES)


def _difficulty_score(sentences: List[str]) -> float:
    """
    Scan sentences containing the skill and return a continuous difficulty score.
    Longer phrases are tested first (more specific → less risk of false matches).
    Returns 0.5 (neutral / Intermediate equivalent) if no modifier is found.
    """
    combined = " ".join(sentences).lower()
    for phrase, weight in sorted(DIFFICULTY_WEIGHTS.items(),
                                  key=lambda x: len(x[0]), reverse=True):
        if phrase in combined:
            return weight
    return 0.5   # default: no modifier found → neutral


def _find_skill_sentences(skill_aliases: List[str],
                           sentences: List[str]) -> List[str]:
    """Return all sentences that contain at least one alias of this skill."""
    matched = []
    for sent in sentences:
        sent_lower = sent.lower()
        if any(re.search(r'(?<![\w/-])' + re.escape(a) + r'(?![\w/-])',
                         sent_lower)
               for a in skill_aliases):
            matched.append(sent)
    return matched


# ── SkillResult dataclass ─────────────────────────────────────────────────────

@dataclass
class SkillResult:
    canonical_name:      str
    source:              str    # "Dictionary" | "ML_NER_Discovered"
    category:            str
    tf_score:            float  # 0–100: how often skill appears in JD
    position_score:      float  # 0–100: how often it appears in Requirements block
    signal_score:        float  # 0–100: strength of importance signals in context
    relevance_score:     float  # 0–100: weighted combination of the three above
    difficulty_score:    float  # 0–100: continuous from JD context modifiers
    is_matched:          bool
    found_in_sentence:   str


# ── TrueHybridMatcher ─────────────────────────────────────────────────────────

class TrueHybridMatcher:
    """
    Dual-engine skill matcher with continuous relevance + difficulty scoring.

    Engine 1 — Alias dictionary (high precision, handles known skills + variants)
    Engine 2 — ML NER model (discovers OOV skills like Snowflake, Looker)

    Scoring (all numbers come from the JD text itself, no pre-assigned weights):
      relevance_score = tf × 0.30 + position × 0.35 + signal × 0.35
      difficulty_score = continuous weight from context modifier words

    Resume matching — alias-normalised on both sides (not exact string match).
    """

    def __init__(self, nlp_base, nlp_ner):
        self.nlp_base    = nlp_base
        self.nlp_ner     = nlp_ner
        self.taxonomy    = TAXONOMY
        self.alias_lookup = ALIAS_LOOKUP

    # ── Engine 1: alias dictionary matching ──────────────────────────────────
    def _extract_via_dictionary(self, jd_lower: str) -> Dict[str, dict]:
        """
        Scan JD text against the full alias lookup table.
        Longer aliases are tested first to avoid "ml" matching inside "mlflow".
        Returns {canonical_lower: {name, source, aliases}} for every hit.
        """
        found = {}
        sorted_aliases = sorted(self.alias_lookup.keys(), key=len, reverse=True)
        for alias in sorted_aliases:
            pattern = r'(?<![\w/-])' + re.escape(alias) + r'(?![\w/-])'
            if re.search(pattern, jd_lower):
                canonical = self.alias_lookup[alias]
                key = canonical.lower()
                if key not in found:
                    found[key] = {
                        "name":    canonical,
                        "source":  "Dictionary",
                        "aliases": [canonical.lower()] + [
                            a.lower()
                            for a in self.taxonomy.get(canonical, {}).get("aliases", [])
                        ],
                        "category": self.taxonomy.get(canonical, {}).get("category", "Other"),
                    }
        return found

    # ── Engine 2: ML NER for OOV discovery ───────────────────────────────────
    def _extract_via_ner(self, jd_text: str,
                          existing: Dict[str, dict]) -> Dict[str, dict]:
        """
        Run the custom NER model. Any entity NOT already found by the dictionary
        is added as an ML-discovered skill.
        Engine 1 takes priority — Engine 2 only fills the gaps.
        """
        doc = self.nlp_ner(jd_text)
        for ent in doc.ents:
            if ent.label_ not in ["HARD_SKILL", "PRODUCT", "ORG"]:
                continue
            canonical = ent.text.strip().title()
            key = canonical.lower()
            if key not in existing:
                existing[key] = {
                    "name":     canonical,
                    "source":   "ML_NER_Discovered",
                    "aliases":  [key],          # no alias table for OOV skills
                    "category": "ML_Discovered",
                }
        return existing

    # ── Relevance + Difficulty scoring ───────────────────────────────────────
    def _score_skill(self,
                     skill_data: dict,
                     sentences: List[str],
                     n_sents: int) -> Dict[str, float]:
        """
        Compute continuous relevance and difficulty scores for one skill.

        Relevance — three components, all from JD text:
          tf_score       = log(1 + mentions) / log(1 + total_sentences)
                           Captures how prominently the skill appears.
                           log compression prevents a skill mentioned 10 times
                           from dominating over one mentioned 3 times.

          position_score = mentions_in_requirements / total_mentions
                           A skill in the Requirements block is more critical
                           than one in the Responsibilities section.

          signal_score   = mean(max importance weight per skill sentence)
                           "Python is required" (1.0) >> "Python experience a plus" (0.12)
                           Takes the mean so a skill in 3 important sentences
                           still scores higher than one in 1 important sentence.

          relevance_score = (tf×0.30 + position×0.35 + signal×0.35) × 100

        Difficulty — one continuous value from context modifier words:
          Scans all sentences containing the skill for modifier phrases.
          Longer phrases are tested first (more specific).
          "5+ years of expert-level Python" → "5+ years" hits first → 0.82
          No modifier found → 0.5 (neutral default).
          difficulty_score = modifier_weight × 100
        """
        skill_sents   = _find_skill_sentences(skill_data["aliases"], sentences)
        mention_count = len(skill_sents)

        if mention_count == 0:
            return {"tf": 0.0, "position": 0.0, "signal": 0.0,
                    "relevance": 0.0, "difficulty": 50.0, "sentence": ""}

        # TF component
        tf = math.log1p(mention_count) / math.log1p(n_sents)

        # Position component
        req_count = sum(1 for s in skill_sents if _is_requirements_sentence(s))
        position  = req_count / mention_count

        # Signal component
        signal = sum(_sentence_signal_score(s) for s in skill_sents) / mention_count

        relevance   = round(min(100.0, (tf*0.30 + position*0.35 + signal*0.35) * 100), 2)
        difficulty  = round(_difficulty_score(skill_sents) * 100, 2)

        return {
            "tf":         round(tf * 100, 2),
            "position":   round(position * 100, 2),
            "signal":     round(signal * 100, 2),
            "relevance":  relevance,
            "difficulty": difficulty,
            "sentence":   skill_sents[0],
        }

    # ── Resume matching (alias-normalised, not exact match) ──────────────────
    def _resume_skill_set(self, resume_text: str) -> set:
        """
        Parse resume through the same alias lookup so both sides use
        canonical names.  "sklearn" → "Scikit-learn", "k8s" → "Kubernetes".
        """
        text_lower    = resume_text.lower()
        found         = set()
        sorted_aliases = sorted(self.alias_lookup.keys(), key=len, reverse=True)
        for alias in sorted_aliases:
            pattern = r'(?<![\w/-])' + re.escape(alias) + r'(?![\w/-])'
            if re.search(pattern, text_lower):
                found.add(self.alias_lookup[alias])
        return found

    # ── Main entry point ─────────────────────────────────────────────────────
    def process(self, jd_text: str, resume_text: str) -> List[SkillResult]:
        """
        Run both engines, score every extracted skill, match against resume.
        Returns a list of SkillResult sorted by relevance_score descending.
        """
        jd_lower  = jd_text.lower()
        doc       = self.nlp_base(jd_text)
        sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
        n_sents   = max(len(sentences), 1)

        # Engine 1 + Engine 2
        extracted = self._extract_via_dictionary(jd_lower)
        extracted = self._extract_via_ner(jd_text, extracted)

        # Resume canonical skill set (alias-normalised)
        resume_skills = self._resume_skill_set(resume_text)

        results = []
        for key, data in extracted.items():
            scores = self._score_skill(data, sentences, n_sents)
            results.append(SkillResult(
                canonical_name    = data["name"],
                source            = data["source"],
                category          = data["category"],
                tf_score          = scores["tf"],
                position_score    = scores["position"],
                signal_score      = scores["signal"],
                relevance_score   = scores["relevance"],
                difficulty_score  = scores["difficulty"],
                is_matched        = data["name"] in resume_skills,
                found_in_sentence = scores["sentence"],
            ))

        return sorted(results, key=lambda x: x.relevance_score, reverse=True)

    def priority(self, r: SkillResult) -> str:
        """
        Combined urgency label from relevance + difficulty.
        urgency = relevance × 0.65 + difficulty × 0.35
        (difficulty contributes because hard skills take longer to acquire)
        """
        urgency = r.relevance_score * 0.65 + r.difficulty_score * 0.35
        if urgency >= 78:   return "🔴 Critical"
        elif urgency >= 58: return "🟠 High"
        elif urgency >= 38: return "🟡 Medium"
        else:               return "🟢 Low"


### Phase 3: Ingesting Complex Data
Notice in this Job Description, we added **"Snowflake"** and **"Looker"**. 
These do NOT exist in the ERA Dictionary. Let's see if the `ML_NER` engine can catch them!

In [5]:
start_time = time.time()

# JD containing both Dictionary skills (Tableau, Excel) and OOV skills (Snowflake, Looker)
sample_jd = """
Analyze large volumes of client data to inform strategies.
Synthesize complex data analytics into easily understood concepts by creating visualizations.
Develop and optimize dashboards using Tableau and Excel.
Must be an expert in Snowflake data warehousing and proficient in Looker for advanced BI reporting.
"""

sample_resume = """
Data Analyst with 3 years of experience. 
Skilled in creating dashboards using Tableau and Excel. 
Basic knowledge of SQL.
"""

print("✅ Complex Test Data Loaded.")

✅ Complex Test Data Loaded.


###  Phase 4: Execution & Gap Analysis Output
We format the output to clearly show the user what they have, what they are missing, and *how* the system found the skill (showing the power of the dual-engine).

In [6]:
matcher = TrueHybridMatcher(nlp_base, nlp_custom_ner)
results = matcher.process(jd_text=sample_jd, resume_text=sample_resume)

matched_skills = [r for r in results if r.is_matched]
missing_skills = [r for r in results if not r.is_matched]

print("\n" + "="*100)
print("✅ MATCHED SKILLS")
print(f"  {'Skill':<20} {'Category':<18} {'Relevance':>9} {'Difficulty':>10} {'Source'}")
print("-"*100)
for r in matched_skills:
    print(f"  {r.canonical_name:<20} {r.category:<18} {r.relevance_score:>9.1f} "
          f"{r.difficulty_score:>10.1f}   {r.source}")

print("\n❌ MISSING SKILLS — ranked by relevance (most urgent gap first)")
print(f"  {'Skill':<20} {'Category':<18} {'Relevance':>9} {'Difficulty':>10} {'Priority':<12} {'Source'}")
print("-"*100)
for r in missing_skills:
    print(f"  {r.canonical_name:<20} {r.category:<18} {r.relevance_score:>9.1f} "
          f"{r.difficulty_score:>10.1f}   {matcher.priority(r):<12} {r.source}")
print("="*100)



✅ MATCHED SKILLS
  Skill                Category           Relevance Difficulty Source
----------------------------------------------------------------------------------------------------
  Tableau              Visualization           12.9       50.0   Dictionary
  Excel                Tools                   12.9       50.0   Dictionary

❌ MISSING SKILLS — ranked by relevance (most urgent gap first)
  Skill                Category           Relevance Difficulty Priority     Source
----------------------------------------------------------------------------------------------------
  Snowflake            Data Engineering        47.9       65.0   🟡 Medium     Dictionary
  GCP                  Cloud                   47.9       65.0   🟡 Medium     Dictionary
  Looker               ML_Discovered           47.9       65.0   🟡 Medium     ML_NER_Discovered
  Bi                   ML_Discovered           47.9       65.0   🟡 Medium     ML_NER_Discovered
  Data Analysis        Analytics         

### 📊 Phase 5: Model Evaluation Metrics (F1-Score Pipeline)
To ensure our Hybrid Model is accurate, we evaluate its extraction capability against a simulated human-annotated `Ground Truth`. We calculate **Precision**, **Recall**, and **F1-Score**.

In [8]:
# Ground Truth: What a human expert says is actually required in the JD
ground_truth_skills: Set[str] = {"Tableau", "Excel", "Data Analytics", "Snowflake", "Looker"}

# What our Dual-Engine extracted
extracted_skills: Set[str] = {r.canonical_name for r in results}

ground_truth_skills = {s.lower() for s in ground_truth_skills}
extracted_skills = {s.lower() for s in extracted_skills}
# Calculate Confusion Matrix Elements
true_positives = len(ground_truth_skills.intersection(extracted_skills))
false_positives = len(extracted_skills - ground_truth_skills)
false_negatives = len(ground_truth_skills - extracted_skills)

# Metrics Calculation
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"📊 Dual-Engine Evaluation Metrics:")
print(f"Ground Truth : {ground_truth_skills}")
print(f"Extracted    : {extracted_skills}")
print("-" * 45)
print(f"Precision : {precision:.2f} (Accuracy of the extractions)")
print(f"Recall    : {recall:.2f} (Ability to find OOV skills missed by dict)")
print(f"F1-Score  : {f1_score:.2f} (Harmonic Mean)")

end_time = time.time()
print(f"\n⏱️ Execution time: {end_time - start_time:.4f} seconds")

📊 Dual-Engine Evaluation Metrics:
Ground Truth : {'data analytics', 'tableau', 'looker', 'snowflake', 'excel'}
Extracted    : {'tableau', 'data analysis', 'develop', 'looker', 'bi', 'snowflake', 'synthesize', 'gcp', 'excel'}
---------------------------------------------
Precision : 0.44 (Accuracy of the extractions)
Recall    : 0.80 (Ability to find OOV skills missed by dict)
F1-Score  : 0.57 (Harmonic Mean)

⏱️ Execution time: 177.2429 seconds
